In [1]:
import os
os.environ["OPENCV_LOG_LEVEL"] = "FATAL"
import cv2
import threading
from socket import *
import time
import subprocess
from flask import Flask, Response, send_file  # 增加了 send_file 用于回传文件
import logging
from LOBOROBOT import LOBOROBOT

In [2]:
# ================= 1. 初始化硬件 =================
print("初始化底盘与摄像头...")
bot = LOBOROBOT()
bot.t_stop(0) # 确保开机静止
# 初始化云台角度
PAN_CH, TILT_CH = 10, 9
current_pan, current_tilt = 80, 0
bot.set_servo_angle(PAN_CH, current_pan)
bot.set_servo_angle(TILT_CH, current_tilt)

初始化底盘与摄像头...


In [3]:
# ================= 2. 视频流全局机制 =================
global_frame = None
frame_lock = threading.Lock()

is_capturing = False      # 拍照状态
camera_ready = False      # 摄像头是否已经打开


def create_camera():
    """
    创建视频流摄像头对象
    """
    gstreamer_pipeline = (
        "libcamerasrc ! video/x-raw, width=1280, height=720, framerate=20/1 ! "
        "videoconvert ! video/x-raw, format=BGR ! appsink drop=true max-buffers=1"
    )
    # 创建一个视频捕获对象cap
    cap = cv2.VideoCapture(gstreamer_pipeline, cv2.CAP_GSTREAMER)

    if not cap.isOpened():
        print("❌ 视频摄像头打开失败")
    else:
        print("✅ 视频摄像头打开成功")

    return cap


def video_capture_thread():
    global global_frame, is_capturing, camera_ready

    cap = create_camera()
    camera_ready = cap.isOpened()

    while True:
        # 如果正在拍高清照，释放摄像头，避免 libcamera-still 抢占失败
        if is_capturing:
            if cap is not None and cap.isOpened():
                print("⏸️ 暂停视频流，释放摄像头给高清拍照...")
                cap.release() # 释放视频流占用的摄像头
                camera_ready = False

            time.sleep(0.1)
            continue # 跳过本次循环剩余代码，重新进入下一次循环

        # 如果拍照结束后摄像头未打开，则重新打开
        if cap is None or not cap.isOpened():
            print("▶️ 正在恢复视频摄像头...")
            cap = create_camera()
            camera_ready = cap.isOpened()
            time.sleep(0.5)
            continue  # 初始化后，跳过本次循环剩余代码，进入下一轮循环以读取数据

        ret, frame = cap.read()

        if ret:
            frame = cv2.flip(frame, -1)

            encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), 82] # 设置JPEG压缩质量的参数
            # 将捕获的原始图像矩阵压缩为.jpg格式
            ret_enc, buffer = cv2.imencode('.jpg', frame, encode_param)

            if ret_enc:
                with frame_lock:
                    global_frame = buffer.tobytes() # 转换成Python标准的bytes对象，便于网络传输
        else:
            time.sleep(0.02)


t_cam = threading.Thread(target=video_capture_thread)
t_cam.daemon = True
t_cam.start()

[1:30:00.867236497] [10635]  INFO Camera camera_manager.cpp:299 libcamera v0.0.4+22-923f5d70
[1:30:00.907700487] [10637]  INFO RPI raspberrypi.cpp:1476 Registered camera /base/soc/i2c0mux/i2c@1/ov5647@36 to Unicam device /dev/media0 and ISP device /dev/media2
[1:30:00.913805278] [10640]  INFO Camera camera.cpp:1028 configuring streams: (0) 1280x720-NV21
[1:30:00.914228978] [10637]  INFO RPI raspberrypi.cpp:851 Sensor: /base/soc/i2c0mux/i2c@1/ov5647@36 - Selected sensor format: 1920x1080-SGBRG10_1X10 - Selected unicam format: 1920x1080-pGAA


In [4]:
# ================= 3. Flask 服务器 (带拍照接口) =================
app = Flask(__name__)
log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)

# 接口1：实时视频流
@app.route('/mycamera')
def video_feed():
    def generate_stream():
        """视频流生成器，推送最新的 global_frame"""
        while True:
            with frame_lock:
                jpeg = global_frame
            if jpeg is not None:
                yield (b'--frame\r\n'
                       b'Content-Type: image/jpeg\r\n\r\n' + jpeg + b'\r\n')
            time.sleep(0.04)
    return Response(generate_stream(), mimetype='multipart/x-mixed-replace; boundary=frame')

# 接口2：高清晰度拍照接口
@app.route('/capture')
def capture_photo():
    global is_capturing

    print("📸 收到远程高清拍照请求...")

    try:
        # 通知视频线程释放摄像头
        is_capturing = True

        # 拍照前停车，减少画面抖动
        bot.t_stop(0)

        # 给视频线程一点时间释放摄像头
        time.sleep(1.0)

        temp_file = "/tmp/high_res_crack.jpg"

        # 如果之前有旧照片，先删除
        if os.path.exists(temp_file):
            os.remove(temp_file)

        cmd = [
            "libcamera-still",
            "-o", temp_file,
            "--immediate",
            "--nopreview",
            "--width", "2880",
            "--height", "1620",
            "--hflip",
            "--vflip",
            "-q", "95"
        ]
        # 打印执行的命令
        print("📷 执行命令：", " ".join(cmd))
        # 执行libcamera-still拍照命令
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=10
        )

        if result.returncode != 0:
            print("❌ libcamera-still 执行失败")
            print("stdout:", result.stdout)
            print("stderr:", result.stderr)
            return f"libcamera-still failed:\n{result.stderr}", 500

        if not os.path.exists(temp_file):
            print("❌ 拍照失败：照片文件未生成")
            return "拍照失败：照片文件未生成", 500

        if os.path.getsize(temp_file) == 0:
            print("❌ 拍照失败：照片文件大小为 0")
            return "拍照失败：照片文件为空", 500

        print("✅ 高清照片拍摄成功，开始发送给上位机")
        return send_file(temp_file, mimetype='image/jpeg')

    except subprocess.TimeoutExpired:
        print("❌ 拍照超时")
        return "拍照超时", 500

    except Exception as e:
        print("❌ 拍照接口异常：", str(e))
        return f"Error: {str(e)}", 500

    finally:
        is_capturing = False
        print("▶️ 拍照结束，准备恢复视频流")

✅ 视频摄像头打开成功


In [5]:
# ================= 4. 网络启动 =================
def get_ip_address():
    try:
        s = socket(AF_INET, SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        ip = s.getsockname()[0]
        s.close()
        return ip
    except:
        return "127.0.0.1"

ip = get_ip_address()
print(f"✅ 树莓派就绪！")
print(f"✅ 视频流地址: http://{ip}:8080/mycamera")
print(f"✅ 拍照下载地址: http://{ip}:8080/capture")

def run_flask():
    app.run(host='0.0.0.0', port=8080, threaded=True, use_reloader=False)

t_flask = threading.Thread(target=run_flask)
t_flask.daemon = True
t_flask.start()

✅ 树莓派就绪！
✅ 视频流地址: http://192.168.43.10:8080/mycamera
✅ 拍照下载地址: http://192.168.43.10:8080/capture
 * Serving Flask app "__main__" (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: off


In [ ]:
# ================= 5. UDP 控制循环 =================
udp_server = socket(AF_INET, SOCK_DGRAM) # 创建一个UDP协议的IPv4套接字
udp_server.bind(('0.0.0.0', 2001)) # 将套接字绑定到本地2001端口，接收发送到该端口的UDP数据包

speed = 50
try:
    while True:
        data_recv, addr = udp_server.recvfrom(1024)
        cmd = data_recv.decode('utf-8').strip()

        if cmd == "UP":          bot.t_up(speed, 0)
        elif cmd == "DOWN":      bot.t_down(speed, 0)
        elif cmd == "LEFT_MOVE": bot.moveLeft(speed, 0)
        elif cmd == "RIGHT_MOVE":bot.moveRight(speed, 0)
        elif cmd == "TURN_L":    bot.turnLeft(speed, 0)
        elif cmd == "TURN_R":    bot.turnRight(speed, 0)
        elif cmd == "UP_LEFT":   bot.forward_Left(speed, 0)
        elif cmd == "UP_RIGHT":  bot.forward_Right(speed,0)
        elif cmd == "DOWN_LEFT": bot.backward_Left(speed,0)
        elif cmd == "DOWN_RIGHT":bot.backward_Right(speed,0)
        elif cmd == "STOP":      bot.t_stop(0)
        elif cmd == "CAM_UP":
            current_tilt = max(0, current_tilt - 10)
            bot.set_servo_angle(TILT_CH, current_tilt)
        elif cmd == "CAM_DOWN":
            current_tilt = min(180, current_tilt + 10)
            bot.set_servo_angle(TILT_CH, current_tilt)
        elif cmd == "CAM_LEFT":
            current_pan = min(180, current_pan + 10)
            bot.set_servo_angle(PAN_CH, current_pan)
        elif cmd == "CAM_RIGHT":
            current_pan = max(0, current_pan - 10)
            bot.set_servo_angle(PAN_CH, current_pan)

except KeyboardInterrupt:
    print("终止")
finally:
    bot.t_stop(0)
    udp_server.close()

📸 收到远程高清拍照请求...
⏸️ 暂停视频流，释放摄像头给高清拍照...
📷 执行命令： libcamera-still -o /tmp/high_res_crack.jpg --immediate --nopreview --width 1920 --height 1080 -q 95
✅ 高清照片拍摄成功，开始发送给上位机
▶️ 拍照结束，准备恢复视频流
▶️ 正在恢复视频摄像头...


[1:30:25.909235233] [10640]  INFO Camera camera_manager.cpp:299 libcamera v0.0.4+22-923f5d70
[1:30:25.946992632] [10703]  INFO RPI raspberrypi.cpp:1476 Registered camera /base/soc/i2c0mux/i2c@1/ov5647@36 to Unicam device /dev/media0 and ISP device /dev/media2
[1:30:25.952142284] [10640]  INFO Camera camera.cpp:1028 configuring streams: (0) 1280x720-NV21
[1:30:25.952683982] [10703]  INFO RPI raspberrypi.cpp:851 Sensor: /base/soc/i2c0mux/i2c@1/ov5647@36 - Selected sensor format: 1920x1080-SGBRG10_1X10 - Selected unicam format: 1920x1080-pGAA


✅ 视频摄像头打开成功
📸 收到远程高清拍照请求...
⏸️ 暂停视频流，释放摄像头给高清拍照...
📷 执行命令： libcamera-still -o /tmp/high_res_crack.jpg --immediate --nopreview --width 1920 --height 1080 -q 95
✅ 高清照片拍摄成功，开始发送给上位机
▶️ 拍照结束，准备恢复视频流
▶️ 正在恢复视频摄像头...


[1:38:47.242803512] [10640]  INFO Camera camera_manager.cpp:299 libcamera v0.0.4+22-923f5d70
[1:38:47.284624263] [11563]  INFO RPI raspberrypi.cpp:1476 Registered camera /base/soc/i2c0mux/i2c@1/ov5647@36 to Unicam device /dev/media0 and ISP device /dev/media2
[1:38:47.291271830] [10640]  INFO Camera camera.cpp:1028 configuring streams: (0) 1280x720-NV21
[1:38:47.292372873] [11563]  INFO RPI raspberrypi.cpp:851 Sensor: /base/soc/i2c0mux/i2c@1/ov5647@36 - Selected sensor format: 1920x1080-SGBRG10_1X10 - Selected unicam format: 1920x1080-pGAA


✅ 视频摄像头打开成功
📸 收到远程高清拍照请求...
⏸️ 暂停视频流，释放摄像头给高清拍照...
📷 执行命令： libcamera-still -o /tmp/high_res_crack.jpg --immediate --nopreview --width 1920 --height 1080 -q 95
✅ 高清照片拍摄成功，开始发送给上位机
▶️ 拍照结束，准备恢复视频流
▶️ 正在恢复视频摄像头...


[1:39:16.992799228] [10640]  INFO Camera camera_manager.cpp:299 libcamera v0.0.4+22-923f5d70
[1:39:17.093330778] [11626]  INFO RPI raspberrypi.cpp:1476 Registered camera /base/soc/i2c0mux/i2c@1/ov5647@36 to Unicam device /dev/media0 and ISP device /dev/media2
[1:39:17.105887217] [10640]  INFO Camera camera.cpp:1028 configuring streams: (0) 1280x720-NV21
[1:39:17.106683652] [11626]  INFO RPI raspberrypi.cpp:851 Sensor: /base/soc/i2c0mux/i2c@1/ov5647@36 - Selected sensor format: 1920x1080-SGBRG10_1X10 - Selected unicam format: 1920x1080-pGAA


✅ 视频摄像头打开成功
📸 收到远程高清拍照请求...
⏸️ 暂停视频流，释放摄像头给高清拍照...
📷 执行命令： libcamera-still -o /tmp/high_res_crack.jpg --immediate --nopreview --width 1920 --height 1080 -q 95
✅ 高清照片拍摄成功，开始发送给上位机
▶️ 拍照结束，准备恢复视频流
▶️ 正在恢复视频摄像头...


[1:40:41.535907051] [10640]  INFO Camera camera_manager.cpp:299 libcamera v0.0.4+22-923f5d70
[1:40:41.573691657] [11773]  INFO RPI raspberrypi.cpp:1476 Registered camera /base/soc/i2c0mux/i2c@1/ov5647@36 to Unicam device /dev/media0 and ISP device /dev/media2
[1:40:41.579166292] [10640]  INFO Camera camera.cpp:1028 configuring streams: (0) 1280x720-NV21
[1:40:41.579696193] [11773]  INFO RPI raspberrypi.cpp:851 Sensor: /base/soc/i2c0mux/i2c@1/ov5647@36 - Selected sensor format: 1920x1080-SGBRG10_1X10 - Selected unicam format: 1920x1080-pGAA


✅ 视频摄像头打开成功
